In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity 
import re
from typing import List, Dict
from dataclasses import dataclass

In [2]:
@dataclass
class MatchResult:
    """Result of resume-to-JD matching"""
    resume_name: str
    match_score: float  # 0-100
    matched_skills: List[str]
    missing_skills: List[str]
    explanation: str

class ResumeParser:
    """Extract skills and experience from resume text"""
    
    # Common skills database
    SKILLS_DB = {
        'python', 'java', 'javascript', 'c++', 'c#', 'php', 'ruby', 'go', 'rust', 'kotlin',
        'sql', 'mongodb', 'postgresql', 'mysql', 'redis', 'elasticsearch', 'DSA',
        'react', 'angular', 'vue', 'django', 'flask', 'fastapi', 'spring', 'express',
        'aws', 'azure', 'gcp', 'docker', 'kubernetes', 'terraform',
        'git', 'jenkins', 'gitlab', 'github', 'ci/cd',
        'machine learning', 'deep learning', 'nlp', 'computer vision', 'tensorflow', 'pytorch', 'keras',
        'html', 'css', 'sass', 'webpack', 'npm', 'rest api', 'graphql',
        'agile', 'scrum', 'jira', 'confluence', 'asana',
        'data analysis', 'data visualization', 'tableau', 'power bi', 'excel', 'pandas',
        'bash', 'shell', 'powershell', 'windows', 'linux', 'macos', 'unix',
        'communication skill', 'leadership', 'teamwork', 'problem solving'
    }
    
    def __init__(self, custom_skills: List[str] = None):
        self.skills_db = self.SKILLS_DB.copy()
        if custom_skills:
            self.skills_db.update(set(custom_skills))
    
    def extract_skills(self, text: str) -> List[str]:
        """Extract skills from resume text"""
        text_lower = text.lower()
        found_skills = []
        
        for skill in self.skills_db:
            if re.search(r'\b' + re.escape(skill) + r'\b', text_lower):
                found_skills.append(skill)
        
        return list(set(found_skills))  # Remove duplicates
    
    def parse(self, resume_text: str) -> Dict:
        """Parse resume and extract components"""
        skills = self.extract_skills(resume_text)
        return {
            'skills': skills,
            'text': resume_text,
            'skill_count': len(skills)
        }

In [3]:
class ResumeMatcher:
    """Match resumes against job description using TF-IDF + Cosine Similarity"""
    
    def __init__(self):
        self.vectorizer = TfidfVectorizer(
            max_features=500,
            stop_words='english',
            ngram_range=(1, 2),  # Unigrams and bigrams
            lowercase=True
        )
        self.parser = ResumeParser()
    
    def match_resume(self, jd_text: str, resume_text: str, resume_name: str = "Resume") -> MatchResult:
        """
        Match a single resume against JD
        
        Args:
            jd_text: Job Description text
            resume_text: Resume text
            resume_name: Name of the resume for identification
        
        Returns:
            MatchResult object with score and details
        """
        # Parse both texts
        jd_parsed = self.parser.parse(jd_text)
        resume_parsed = self.parser.parse(resume_text)
        
        jd_skills = set(jd_parsed['skills'])
        resume_skills = set(resume_parsed['skills'])
        
        # Calculate TF-IDF similarity
        try:
            tfidf_matrix = self.vectorizer.fit_transform([jd_text, resume_text])
            similarity = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
            tfidf_score = similarity * 100  # Convert to 0-100 scale
        except:
            tfidf_score = 0
        
        # Calculate skill-based score
        if len(jd_skills) > 0:
            matched = resume_skills.intersection(jd_skills)
            skill_score = (len(matched) / len(jd_skills)) * 100
        else:
            matched = set()
            skill_score = 0
        
        # Combined score (60% skill match, 40% content similarity)
        final_score = (skill_score * 0.6) + (tfidf_score * 0.4)
        final_score = min(100, final_score)  # Cap at 100
        
        # Determine missing skills
        missing_skills = sorted(list(jd_skills - resume_skills))
        matched_skills = sorted(list(matched))
        
        # Generate explanation
        explanation = self._generate_explanation(
            final_score, len(matched_skills), len(missing_skills), len(jd_skills)
        )
        
        return MatchResult(
            resume_name=resume_name,
            match_score=round(final_score, 2),
            matched_skills=matched_skills,
            missing_skills=missing_skills,
            explanation=explanation
        )
    
    def match_multiple_resumes(
        self, jd_text: str, resumes: List[Dict[str, str]]
    ) -> List[MatchResult]:
        """
        Match multiple resumes against JD
        
        Args:
            jd_text: Job Description text
            resumes: List of dicts with 'name' and 'text' keys
        
        Returns:
            List of MatchResult objects
        """
        results = []
        for resume in resumes:
            result = self.match_resume(
                jd_text,
                resume['text'],
                resume.get('name', 'Resume')
            )
            results.append(result)
        
        # Sort by score descending
        results.sort(key=lambda x: x.match_score, reverse=True)
        return results
    
    def _generate_explanation(
        self, score: float, matched_count: int, missing_count: int, total_required: int
    ) -> str:
        """Generate human-readable explanation"""
        if score >= 80:
            return f"Excellent match! {matched_count}/{total_required} required skills present. Only {missing_count} skills needed."
        elif score >= 60:
            return f"Good match. Candidate has {matched_count}/{total_required} required skills. Missing {missing_count} important skill(s)."
        elif score >= 40:
            return f"Moderate match. {matched_count}/{total_required} skills found. Candidate needs training in {missing_count} area(s)."
        else:
            return f"Low match. Only {matched_count}/{total_required} required skills present. Significant skill gap in {missing_count} area(s)."

In [5]:
from PyPDF2 import PdfReader

def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract text from a PDF file"""
    text = ""
    try:
        pdf_reader = PdfReader(pdf_path)
        for page in pdf_reader.pages:
            text += page.extract_text()
    except Exception as e:
        print(f"Error reading PDF: {e}")
    return text

def process_resume_pdf(pdf_path: str, jd_text: str, resume_name: str = None) -> MatchResult:
    """
    Extract text from resume PDF and match it against job description
    
    Args:
        pdf_path: Path to the resume PDF file
        jd_text: Job Description text
        resume_name: Optional name for the resume (defaults to filename)
    
    Returns:
        MatchResult object with match score and details
    """
    # Extract text from PDF
    resume_text = extract_text_from_pdf(pdf_path)
    
    # Use filename as resume name if not provided
    if resume_name is None:
        resume_name = pdf_path.split('/')[-1]
    
    # Match resume against JD
    matcher = ResumeMatcher()
    result = matcher.match_resume(jd_text, resume_text, resume_name)
    
    return result

# Example usage:
# jd = "We are looking for a Python developer with Django, AWS, and PostgreSQL experience..."
# result = process_resume_pdf("path/to/resume.pdf", jd)
# print(f"Match Score: {result.match_score}")
# print(f"Matched Skills: {result.matched_skills}")
# print(f"Missing Skills: {result.missing_skills}")
# print(f"Explanation: {result.explanation}")

In [12]:
# Input job description
job_description = """
Position: Senior Python Developer

We are looking for an experienced Python Developer with strong skills in:
- Python and Django/FastAPI
- PostgreSQL and MongoDB
- AWS and Docker
- REST API design
- Git and CI/CD pipelines

Requirements:
- 5+ years of Python development experience
- Strong understanding of software design patterns
- Experience with machine learning libraries (scikit-learn, pandas)
- Excellent problem-solving skills
- Team leadership and communication abilities

Nice to have:
- Kubernetes experience
- GraphQL knowledge
- Terraform for Infrastructure as Code
"""

print("Job Description:")
print("=" * 100)
print(job_description)

Job Description:

Position: Senior Python Developer

We are looking for an experienced Python Developer with strong skills in:
- Python and Django/FastAPI
- PostgreSQL and MongoDB
- AWS and Docker
- REST API design
- Git and CI/CD pipelines

Requirements:
- 5+ years of Python development experience
- Strong understanding of software design patterns
- Experience with machine learning libraries (scikit-learn, pandas)
- Excellent problem-solving skills
- Team leadership and communication abilities

Nice to have:
- Kubernetes experience
- GraphQL knowledge
- Terraform for Infrastructure as Code



In [14]:
# Sample resumes for matching
resumes = [
    {
        "name": "candidate_1.txt",
        "text": """
        Senior Python Developer
        
        Experience:
        - 7 years of Python development with Django and FastAPI
        - Strong expertise in PostgreSQL and MongoDB databases
        - AWS cloud architecture and Docker containerization
        - RESTful API design and GraphQL implementation
        - Git version control and CI/CD pipeline setup with Jenkins
        - Machine learning projects using pandas, scikit-learn
        - Leadership of 5-person development team
        
        Skills: Python, Django, FastAPI, PostgreSQL, MongoDB, AWS, Docker, REST API, GraphQL, Git, Jenkins, CI/CD, Kubernetes, pandas, scikit-learn, Problem Solving, Leadership
        """
    },
    {
        "name": "candidate_2.txt",
        "text": """
        Mid-level Python Developer
        
        Experience:
        - 3 years of Python development
        - Basic Django experience
        - MySQL database management
        - Docker basics
        - Git and GitHub
        - Excel and data analysis
        
        Skills: Python, Django, MySQL, Docker, Git, GitHub, Excel, Communication
        """
    },
    {
        "name": "candidate_3.txt",
        "text": """
        Senior Full-Stack Developer
        
        Experience:
        - 8 years of full-stack development
        - 5 years with Python backend and React frontend
        - PostgreSQL and Redis expertise
        - AWS services and Terraform for infrastructure
        - Kubernetes orchestration
        - CI/CD with GitLab and GitHub Actions
        - Team leadership and mentoring
        - Machine learning integration with TensorFlow
        
        Skills: Python, React, PostgreSQL, Redis, AWS, Terraform, Kubernetes, GitLab, GitHub, CI/CD, TensorFlow, Leadership, Teamwork
        """
    }
]

# Run matching
matcher = ResumeMatcher()
results = matcher.match_multiple_resumes(job_description, resumes)

# Display results
print("\n" + "="*70)
print("RESUME SCREENING RESULTS")
print("="*70 + "\n")

for i, result in enumerate(results, 1):
    print(f"Rank #{i}: {result.resume_name}")
    print(f"Match Score: {result.match_score}/100")
    print(f"Matched Skills ({len(result.matched_skills)}): {', '.join(result.matched_skills) if result.matched_skills else 'None'}")
    print(f"Missing Skills ({len(result.missing_skills)}): {', '.join(result.missing_skills) if result.missing_skills else 'None'}")
    print(f"Explanation: {result.explanation}")
    print("-" * 70 + "\n")


RESUME SCREENING RESULTS

Rank #1: candidate_1.txt
Match Score: 72.41/100
Matched Skills (15): aws, ci/cd, django, docker, fastapi, git, graphql, kubernetes, leadership, machine learning, mongodb, pandas, postgresql, python, rest api
Missing Skills (1): terraform
Explanation: Good match. Candidate has 15/16 required skills. Missing 1 important skill(s).
----------------------------------------------------------------------

Rank #2: candidate_3.txt
Match Score: 38.14/100
Matched Skills (8): aws, ci/cd, kubernetes, leadership, machine learning, postgresql, python, terraform
Missing Skills (8): django, docker, fastapi, git, graphql, mongodb, pandas, rest api
Explanation: Low match. Only 8/16 required skills present. Significant skill gap in 8 area(s).
----------------------------------------------------------------------

Rank #3: candidate_2.txt
Match Score: 23.52/100
Matched Skills (4): django, docker, git, python
Missing Skills (12): aws, ci/cd, fastapi, graphql, kubernetes, leadersh

In [ ]:
import os
import glob
from pathlib import Path

def process_multiple_pdfs(pdf_directory_or_list, job_description):
    """
    Process multiple PDF resumes and match them against job description
    
    Args:
        pdf_directory_or_list: Either a directory path or list of PDF file paths
        job_description: Job description text to match against
    
    Returns:
        List of MatchResult objects sorted by match score
    """
    # Get PDF file paths
    if isinstance(pdf_directory_or_list, str):
        # It's a directory path
        pdf_files = glob.glob(os.path.join(pdf_directory_or_list, "*.pdf"))
    else:
        # It's a list of file paths
        pdf_files = pdf_directory_or_list
    
    if not pdf_files:
        print("No PDF files found!")
        return []
    
    print(f"Found {len(pdf_files)} PDF file(s) to process...\n")
    
    matcher = ResumeMatcher()
    results = []
    
    for pdf_path in pdf_files:
        print(f"Processing: {os.path.basename(pdf_path)}...", end=" ")
        try:
            resume_text = extract_text_from_pdf(pdf_path)
            
            if not resume_text.strip():
                print("ERROR - No text extracted")
                continue
            
            result = matcher.match_resume(
                job_description,
                resume_text,
                resume_name=os.path.basename(pdf_path)
            )
            results.append(result)
            print(f"✓ Match Score: {result.match_score}")
        
        except Exception as e:
            print(f"ERROR - {str(e)}")
            continue
    
    # Sort by score descending
    results.sort(key=lambda x: x.match_score, reverse=True)
    return results

def export_results_to_csv(results, output_file="resume_matching_results.csv"):
    """Export matching results to CSV file"""
    import csv
    
    try:
        with open(output_file, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow([
                'Rank', 'Resume Name', 'Match Score', 'Matched Skills', 
                'Missing Skills', 'Matched Count', 'Missing Count', 'Explanation'
            ])
            
            for i, result in enumerate(results, 1):
                writer.writerow([
                    i,
                    result.resume_name,
                    result.match_score,
                    '; '.join(result.matched_skills),
                    '; '.join(result.missing_skills),
                    len(result.matched_skills),
                    len(result.missing_skills),
                    result.explanation
                ])
        
        print(f"✓ Results exported to: {output_file}")
        return output_file
    
    except Exception as e:
        print(f"Error exporting to CSV: {e}")
        return None

def display_pdf_results(results):
    """Display PDF matching results in formatted table"""
    if not results:
        print("No results to display")
        return
    
    print("\n" + "="*80)
    print("PDF RESUME SCREENING RESULTS")
    print("="*80 + "\n")
    
    for i, result in enumerate(results, 1):
        print(f"Rank #{i}: {result.resume_name}")
        print(f"Match Score: {result.match_score}/100")
        print(f"Matched Skills ({len(result.matched_skills)}): {', '.join(result.matched_skills) if result.matched_skills else 'None'}")
        print(f"Missing Skills ({len(result.missing_skills)}): {', '.join(result.missing_skills) if result.missing_skills else 'None'}")
        print(f"Explanation: {result.explanation}")
        print("-" * 80 + "\n")

# ============================================================================
# SECTION: Process PDF Resumes
# ============================================================================

# Option 1: Process PDFs from a directory
# Uncomment and set your PDF directory path
pdf_directory = "D:/Download/Mohit_Kumar_Gupta_Resume.pdf"
pdf_results = process_multiple_pdfs(pdf_directory, job_description)
display_pdf_results(pdf_results)
export_results_to_csv(pdf_results)

# Option 2: Process specific PDF files (as a list)
# Uncomment and provide your PDF file paths
# pdf_files = [
#     "path/to/resume1.pdf",
#     "path/to/resume2.pdf",
#     "path/to/resume3.pdf"
# ]
# pdf_results = process_multiple_pdfs(pdf_files, job_description)
# display_pdf_results(pdf_results)
# export_results_to_csv(pdf_results)


print("\n1. From a directory:")
print("   pdf_results = process_multiple_pdfs('pdf_directory', job_description)")
print("   display_pdf_results(pdf_results)")
print("   export_results_to_csv(pdf_results)")
print("\n2. From specific files:")
print("   pdf_files = ['resume1.pdf', 'resume2.pdf']")
print("   pdf_results = process_multiple_pdfs(pdf_files, job_description)")
print("   display_pdf_results(pdf_results)")

No PDF files found!
No results to display
✓ Results exported to: resume_matching_results.csv
✓ PDF processing functions loaded successfully!

To process PDF resumes, use one of these methods:

1. From a directory:
   pdf_results = process_multiple_pdfs('pdf_directory', job_description)
   display_pdf_results(pdf_results)
   export_results_to_csv(pdf_results)

2. From specific files:
   pdf_files = ['resume1.pdf', 'resume2.pdf']
   pdf_results = process_multiple_pdfs(pdf_files, job_description)
   display_pdf_results(pdf_results)
